# Random Forest — voorbereiding en beslisregel

**Concept op verzoek van de gebruiker, uitgewerkt met Codex.** Milan moet
de keuzes en eigen bijdrage zelf beoordelen. Alle nieuwe bestanden blijven onder
`solutions/milan`. We gebruiken de cursusmethoden en de oorspronkelijke elf
invoervelden. Geen nieuwe onafhankelijke eindtest of menselijke review geclaimd.
De dure training is afzonderlijk werkelijk uitgevoerd; onderstaande rapportcellen
    worden van boven naar beneden uitgevoerd; de precieze uitvoeringsmethode staat
in de notebookmetadata en uitvoeringsreceipt.

## Model en zoekruimte
We gebruiken exact dezelfde buitenfolds. Per familie selecteren drie binnenfolds de instellingen. Scaling wordt bij SVM binnen de pipeline geleerd; boommodellen krijgen geen onnodige standaardisatie.

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
HERE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'results_v2/summary.csv').exists())
sys.path.insert(0, str(HERE))
from mushroom_alternative import development_data, scores
from improve import experiments, make_pipeline, Measurements
X, y, ids, folds = development_data()
summary = pd.read_csv(HERE/'results_v2/summary.csv')
details = pd.read_csv(HERE/'results_v2/outer_fold_metrics.csv')
selections = json.loads((HERE/'results_v2/selections.json').read_text())
families = ['rf_compatible', 'rf_features']
for family in families:
    estimator, grid = experiments()[family]
    display(estimator)
    print('Zoekruimte:', grid)


Zoekruimte: {'preprocessor__cat__impute__strategy': ['most_frequent', 'constant'], 'classifier__class_weight': ['balanced', None], 'classifier__min_samples_leaf': [1, 2]}
Zoekruimte: [{'preprocessor': [ColumnTransformer(sparse_threshold=0,
                  transformers=[('num',
                                 Pipeline(steps=[('impute',
                                                  SimpleImputer(strategy='median'))]),
                                 ['cap_diameter', 'stem_height', 'stem_width']),
                                ('cat',
                                 Pipeline(steps=[('impute',
                                                  SimpleImputer(fill_value='missing',
                                                                strategy='constant')),
                                                 ('encode',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse

Pipeline(steps=[('measurements', Measurements()),
                ('preprocessor',
                 ColumnTransformer(sparse_threshold=0,
                                   transformers=[('num',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['cap_diameter',
                                                   'stem_height',
                                                   'stem_width']),
                                                 ('cat',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='most_frequent')),
                                                                  ('encode',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['spore_print_color',
                                                   'gill_color', 'habitat',
                                                   'season', 'ring_type',
                                                   'cap_shape', 'stem_surface',
                                                   'jumbled_noise_1'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced', max_depth=20,
                                        max_features=None, min_samples_split=4,
                                        n_estimators=300, n_jobs=1,
                                        random_state=42))])

Pipeline(steps=[('measurements', Measurements()),
                ('preprocessor',
                 ColumnTransformer(sparse_threshold=0,
                                   transformers=[('num',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['cap_diameter',
                                                   'stem_height',
                                                   'stem_width']),
                                                 ('cat',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='constant')),
                                                                  ('encode',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['spore_print_color',
                                                   'gill_color', 'habitat',
                                                   'season', 'ring_type',
                                                   'cap_shape', 'stem_surface',
                                                   'jumbled_noise_1'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced', max_depth=20,
                                        max_features=None, min_samples_split=4,
                                        n_estimators=300, n_jobs=1,
                                        random_state=42))])

## De werkelijk uitgevoerde selectie
Deze tabel geeft per buitenfold de binnen training gekozen instellingen en drempel. Een binnenscore is een selectiescore, niet de buitenmeting.

In [2]:
display(pd.DataFrame([s for s in selections if s['family'] in families]))
for family in families:
    display(details[details.model.str.startswith(family+'_')])


,family,fold,best_parameters,inner_cv_macro_f1,f1_threshold,seconds
0,rf_compatible,1,"{'classifier__class_weight': 'balanced', 'clas...",0.753301,0.46,66.622589
1,rf_compatible,2,"{'classifier__class_weight': 'balanced', 'clas...",0.751344,0.43,68.875720
2,rf_compatible,3,"{'classifier__class_weight': 'balanced', 'clas...",0.752924,0.46,68.007886
3,rf_compatible,4,"{'classifier__class_weight': 'balanced', 'clas...",0.738770,0.46,70.426322
4,rf_compatible,5,"{'classifier__class_weight': 'balanced', 'clas...",0.749707,0.47,68.614678
5,rf_features,1,"{'classifier__min_samples_leaf': 1, 'measureme...",0.755059,0.47,73.348132
6,rf_features,2,"{'classifier__min_samples_leaf': 3, 'measureme...",0.755809,0.51,80.391904
7,rf_features,3,"{'classifier__min_samples_leaf': 1, 'measureme...",0.757134,0.47,81.803543
8,rf_features,4,"{'classifier__min_samples_leaf': 3, 'measureme...",0.737858,0.49,74.732620
9,rf_features,5,"{'classifier__min_samples_leaf': 1, 'measureme...",0.751689,0.45,80.138116


,model,fold,threshold,accuracy,macro_f1,balanced_accuracy,recall_p,precision_p,specificity_e,roc_auc,average_precision,tn,fp,fn,tp
5,rf_compatible_default,1,0.50,0.784146,0.758547,0.749206,0.604502,0.776860,0.893910,0.814756,0.758368,455,54,123,188
6,rf_compatible_f1_threshold,1,0.46,0.780488,0.759922,0.753763,0.643087,0.743494,0.864440,0.814756,0.758368,440,69,111,200
7,rf_compatible_default,2,0.50,0.774390,0.748080,0.739471,0.594855,0.758197,0.884086,0.824743,0.756937,450,59,126,185
8,rf_compatible_f1_threshold,2,0.43,0.773171,0.758206,0.757251,0.691318,0.704918,0.823183,0.824743,0.756937,419,90,96,215
9,rf_compatible_default,3,0.50,0.780488,0.754672,0.745731,0.603226,0.766393,0.888235,0.825269,0.754746,453,57,123,187
10,rf_compatible_f1_threshold,3,0.46,0.786585,0.766408,0.760120,0.651613,0.750929,0.868627,0.825269,0.754746,443,67,108,202
11,rf_compatible_default,4,0.50,0.825610,0.806284,0.795920,0.674194,0.832669,0.917647,0.859386,0.815701,468,42,101,209
12,rf_compatible_f1_threshold,4,0.46,0.824390,0.809954,0.805060,0.725806,0.792254,0.884314,0.859386,0.815701,451,59,85,225
13,rf_compatible_default,5,0.50,0.780488,0.755530,0.746996,0.609677,0.762097,0.884314,0.805623,0.743915,451,59,121,189
14,rf_compatible_f1_threshold,5,0.47,0.773171,0.751136,0.744908,0.629032,0.733083,0.860784,0.805623,0.743915,439,71,115,195


,model,fold,threshold,accuracy,macro_f1,balanced_accuracy,recall_p,precision_p,specificity_e,roc_auc,average_precision,tn,fp,fn,tp
25,rf_features_default,1,0.50,0.790244,0.765576,0.755993,0.614148,0.786008,0.897839,0.812665,0.756091,457,52,120,191
26,rf_features_f1_threshold,1,0.47,0.776829,0.755730,0.749566,0.636656,0.738806,0.862475,0.812665,0.756091,439,70,113,198
27,rf_features_default,2,0.50,0.775610,0.749220,0.740453,0.594855,0.761317,0.886051,0.827150,0.756317,451,58,126,185
28,rf_features_f1_threshold,2,0.51,0.775610,0.748324,0.739202,0.588424,0.765690,0.889980,0.827150,0.756317,453,56,128,183
29,rf_features_default,3,0.50,0.774390,0.746727,0.737666,0.587097,0.761506,0.888235,0.826660,0.750732,453,57,128,182
30,rf_features_f1_threshold,3,0.47,0.786585,0.764908,0.757590,0.638710,0.758621,0.876471,0.826660,0.750732,447,63,112,198
31,rf_features_default,4,0.50,0.828049,0.809958,0.800411,0.687097,0.828794,0.913725,0.857445,0.814419,466,44,97,213
32,rf_features_f1_threshold,4,0.49,0.828049,0.811194,0.802941,0.700000,0.818868,0.905882,0.857445,0.814419,462,48,93,217
33,rf_features_default,5,0.50,0.781707,0.757929,0.749873,0.619355,0.758893,0.880392,0.801872,0.744919,449,61,118,192
34,rf_features_f1_threshold,5,0.45,0.781707,0.762891,0.758096,0.661290,0.734767,0.854902,0.801872,0.744919,436,74,105,205


## Resultaten tegenover RF300
Standaardlabels en een afgestemde macro-F1-beslisregel staan apart. Die drempel is niet op buitenvalidatie gekozen. Grotere recall heeft alleen betekenis samen met precision, FN en FP.

In [3]:
display(summary[summary.model.eq('rf300_reference_default') | summary.model.str.startswith(tuple(families))][['model','macro_f1_mean','macro_f1_sd','recall_p_mean','precision_p_mean','fn_total','fp_total']])


,model,macro_f1_mean,macro_f1_sd,recall_p_mean,precision_p_mean,fn_total,fp_total
0,rf300_reference_default,0.761876,0.021452,0.604390,0.786176,614,256
1,rf_compatible_default,0.764623,0.023600,0.617291,0.779243,594,271
2,rf_compatible_f1_threshold,0.769125,0.023462,0.668171,0.744936,515,356
5,rf_features_default,0.765882,0.025742,0.620510,0.779304,589,272
6,rf_features_f1_threshold,0.768609,0.024684,0.645016,0.763350,551,311


## Opnieuw trainen
De opgeslagen training herhalen kan met onderstaande vlag. Standaard worden alleen de echte resultaten gelezen, zonder onnodig opnieuw te fitten.

In [4]:
RERUN = False
if RERUN:
    import joblib
    from improve import run
    with joblib.parallel_config(backend='threading'):
        run(families)
else:
    print('Rapportcontrole; geen nieuwe training gestart.')


Rapportcontrole; geen nieuwe training gestart.
